In [1]:
import numpy as np
import pandas as pd
import os
import sys
import time

In [2]:
import sklearn.tree
import sklearn.linear_model
import sklearn.metrics
import sklearn.ensemble

In [3]:

from pretty_print_sklearn_tree import pretty_print_sklearn_tree

In [4]:
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd


plt.style.use('seaborn-v0_8') 
%pip install seaborn --quiet
import seaborn as sns
sns.set('notebook', font_scale=1.25, style='whitegrid')

Note: you may need to restart the kernel to use updated packages.


In [5]:
def add_username_to_plot(username="gracy.garg"):
    plt.text(
        0.95, 0.95,
        username,
        ha='right',
        va='top',
        transform=plt.gca().transAxes,
        fontsize=10, color='gray', alpha=0.7
    )

# Load all data from train/valid/test

In [6]:
FEATURE_COUNT = 7729 
RANDOM_STATE = 101

DATA_DIR = 'data_product_reviews' 

### Load training

In [7]:
from scipy.sparse import csr_matrix

def load_data_and_convert(feature_file, label_file):
    # Load features WITH header (first row is vocabulary)
    X_df = pd.read_csv(os.path.join(DATA_DIR, feature_file), header=0)
    
    # Load labels WITHOUT header
    y_df = pd.read_csv(os.path.join(DATA_DIR, label_file), header=0)
    
    print(f"Features shape: {X_df.shape}")
    print(f"Labels shape: {y_df.shape}")
    
    # They should already match! Don't truncate
    if X_df.shape[0] != y_df.shape[0]:
        raise ValueError(f"Mismatch: {X_df.shape[0]} features vs {y_df.shape[0]} labels")
    
    # Convert to numeric
    X_df = X_df.apply(pd.to_numeric, errors='coerce').fillna(0)
    y_df = y_df.apply(pd.to_numeric, errors='coerce').fillna(0)
    
    X_sparse = csr_matrix(X_df.values.astype(np.float64))
    y_array = y_df.values.ravel().astype(int)
    
    return X_sparse, y_array

In [8]:
X_train, y_train = load_data_and_convert('x_train.csv', 'y_train.csv')
print(f"Train shapes: X={X_train.shape}, y={y_train.shape}")

Features shape: (6346, 7729)
Labels shape: (6346, 1)
Train shapes: X=(6346, 7729), y=(6346,)


### Load validation set

In [9]:
X_val, y_val = load_data_and_convert('x_valid.csv', 'y_valid.csv')
print(f"Validation shapes: X={X_val.shape}, y={y_val.shape}")

Features shape: (792, 7729)
Labels shape: (792, 1)
Validation shapes: X=(792, 7729), y=(792,)


### Load test set 

In [10]:
X_test, y_test = load_data_and_convert('x_test.csv', 'y_test.csv')
print(f"Test shapes: X={X_test.shape}, y={y_test.shape}")

Features shape: (793, 7729)
Labels shape: (793, 1)
Test shapes: X=(793, 7729), y=(793,)


### Load vocabulary as a list of strings

In [11]:
vocab = list(pd.read_csv(os.path.join(DATA_DIR, "x_train.csv"), nrows=1).columns)
import json
with open(os.path.join(DATA_DIR, "vocab.json"), "w", encoding="utf-8") as f:
    json.dump(vocab, f, ensure_ascii=False, indent=2)


In [12]:
# # print vocab
# print("First 20 vocabulary words:")
# for i, word in enumerate(vocab[:20]):
#     print(f"{i:2d}: {word}")


### Pack training and validation sets into big arrays (so we can use sklearn's hyperparameter search tools)

In [13]:
from scipy.sparse import vstack
import numpy as np

X_all = vstack([X_train, X_val])
y_all = np.concatenate([y_train, y_val])
print(f"Combined dataset shape: {X_all.shape}")
print(f"Combined labels shape: {y_all.shape}")


Combined dataset shape: (7138, 7729)
Combined labels shape: (7138,)


In [14]:
# from sklearn.model_selection import PredefinedSplit
# my_splitter = PredefinedSplit(test_fold)

# print(f"\nCombined train+val set: {x_trva_NF.shape[0]} examples")
# print(f"Custom split: {n_train} train, {n_total - n_train} validation")

# Problem 1: Decision Trees

## 1A: Train a simple tree with depth 3

In [15]:
from sklearn.tree import DecisionTreeClassifier
# from DecisionTreeRegressor import MyDecisionTreeClassifier


simple_dt = DecisionTreeClassifier(
    criterion='gini',
    max_depth=3,
    min_samples_leaf=1,
    min_samples_split=2,
    random_state=101
)

# my_simple_dt = MyDecisionTreeClassifier(
#     max_depth=3,
#     min_samples_leaf=1,
#     min_samples_split=2
# )

### **Fit the tree** 

**TODO Train on the training set** in the next coding cell

In [16]:
simple_dt.fit(X_train, y_train)

# Convert sparse matrices to dense arrays
X_train_dense = X_train.toarray()
X_val_dense = X_val.toarray()
X_test_dense = X_test.toarray()
# Then use with custom tree
# my_simple_dt.fit(X_train_dense, y_train)

In [17]:
# feature_names = vocab if 'vocab' in globals() else [f"word_{i}" for i in range(X_train.shape[1])]
# print("Custom Decision Tree Structure:")
# print(my_simple_dt)  # This uses the __str__ method from your custom class

In [18]:
from sklearn.metrics import balanced_accuracy_score

simple_dt_train_acc  = balanced_accuracy_score(y_train, simple_dt.predict(X_train))
simple_dt_valid_acc  = balanced_accuracy_score(y_val, simple_dt.predict(X_val))
simple_dt_test_acc   = balanced_accuracy_score(y_test, simple_dt.predict(X_test))

print(f"Simple DT Train Balanced Accuracy: {simple_dt_train_acc:.4f}")
print(f"Simple DT Valid Balanced Accuracy: {simple_dt_valid_acc:.4f}")
print(f"Simple DT Test Balanced Accuracy: {simple_dt_test_acc:.4f}")

Simple DT Train Balanced Accuracy: 0.6458
Simple DT Valid Balanced Accuracy: 0.6446
Simple DT Test Balanced Accuracy: 0.6458


### **Figure 1: Print Tree** 

Use a helper function from the starter code

In [19]:
feature_names = vocab if 'vocab' in globals() else [f"word_{i}" for i in range(X_train.shape[1])]
pretty_print_sklearn_tree(simple_dt, feature_names)

The binary tree structure has 15 nodes.
- depth   0 has    1 nodes, of which    0 are leaves
- depth   1 has    2 nodes, of which    0 are leaves
- depth   2 has    4 nodes, of which    0 are leaves
- depth   3 has    8 nodes, of which    8 are leaves
The decision tree:  (Note: Y = 'yes' to above question; N = 'no')
Decision: X['great'] <= 0.50?
  Y Decision: X['excel'] <= 0.50?
    Y Decision: X['disappoint'] <= 0.50?
      Y Leaf: p(y=1 | this leaf) = 0.430 (1 total training examples)
      N Leaf: p(y=1 | this leaf) = 0.114 (1 total training examples)
    N Decision: X['disappoint'] <= 0.50?
      Y Leaf: p(y=1 | this leaf) = 0.903 (1 total training examples)
      N Leaf: p(y=1 | this leaf) = 0.429 (1 total training examples)
  N Decision: X['return'] <= 0.50?
    Y Decision: X['bad'] <= 0.50?
      Y Leaf: p(y=1 | this leaf) = 0.745 (1 total training examples)
      N Leaf: p(y=1 | this leaf) = 0.415 (1 total training examples)
    N Decision: X['movie'] <= 0.50?
      Y Leaf: p(y

Yes, there are internal nodes whose two child leaves correspond to the same sentiment class.

For example, consider this portion of the tree:

```
Decision: X['disappoint'] <= 0.50?
  Y  Leaf: p(y=1 | this leaf) = 0.430  → NEGATIVE
  N  Leaf: p(y=1 | this leaf) = 0.114  → NEGATIVE
```

**Analysis:**
- In this example, both child leaves predict the same sentiment class (both < 0.5 → negative), indicating that the `disappoint` split doesn't effectively separate the classes.

This occurs when an internal split provides minimal information gain i.e. *it slightly reduces impurity* (Gini coefficient) but both resulting subsets are still dominated by the same class. The decision tree algorithm accepts these splits because they provide some improvement in the objective function, even if the practical benefit is limited.

## 1B : Find best Decision Tree with grid search

In [20]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier

# Define parameter grid
param_grid = {
    'max_depth': [2, 8, 32, 128],
    'min_samples_leaf': [1, 3, 9],
    'random_state': [101]
}

from sklearn.model_selection import PredefinedSplit

# Create split indices: -1 for train, 0 for validation
test_fold = np.concatenate([
    np.full(X_train.shape[0], -1),  # training indices
    np.zeros(X_val.shape[0])        # validation indices
])

my_splitter = PredefinedSplit(test_fold)


# Initialize base DecisionTree model
tree_model = DecisionTreeClassifier(criterion='gini')

# Run grid search
grid_search = GridSearchCV(
    estimator=tree_model,
    param_grid=param_grid,
    scoring='balanced_accuracy',
    cv=my_splitter,
    return_train_score=True,
    refit=False,
    n_jobs=-1
)

grid_search.fit(X_all, y_all)

# Show best params and scores
print("Best Parameters:", grid_search.best_params_)
print("Best Balanced Accuracy:", grid_search.best_score_)


Best Parameters: {'max_depth': 32, 'min_samples_leaf': 3, 'random_state': 101}
Best Balanced Accuracy: 0.7323798101459631


In [21]:
# Convert GridSearchCV results to a DataFrame
results_df = pd.DataFrame(grid_search.cv_results_)

# Select and rename only the most useful columns
summary_df = results_df[
    [
        "param_max_depth",
        "param_min_samples_leaf",
        "mean_train_score",
        "mean_test_score",
        "rank_test_score",
        "mean_fit_time",
    ]
].sort_values(by="param_max_depth")

# Round values for readability
summary_df = summary_df.round(4)

# Print a clean, formatted table
print(summary_df.to_string(index=False))


 param_max_depth  param_min_samples_leaf  mean_train_score  mean_test_score  rank_test_score  mean_fit_time
               2                       1            0.6404           0.6339               10         0.0450
               2                       3            0.6404           0.6339               10         0.0312
               2                       9            0.6404           0.6339               10         0.0390
               8                       1            0.7265           0.6989                7         0.2633
               8                       3            0.7219           0.6976                8         0.2173
               8                       9            0.7128           0.6962                9         0.2451
              32                       1            0.9182           0.7274                3         1.0523
              32                       3            0.8772           0.7324                1         0.8646
              32            

### Build the best decision tree

**TODO Build the Best Tree on the training set** in the next coding cell



In [22]:
from sklearn.tree import DecisionTreeClassifier

# Build and train the best Decision Tree using the discovered hyperparameters
# The best values are max_depth=32, min_samples_leaf=3 from grid search

best_dt = DecisionTreeClassifier(
    criterion='gini',
    max_depth=grid_search.best_params_['max_depth'],
    min_samples_leaf=grid_search.best_params_['min_samples_leaf'],
    random_state=101
)

best_dt.fit(X_train, y_train)

# Evaluate on all splits
y_train_pred = best_dt.predict(X_train)
y_valid_pred = best_dt.predict(X_val)
y_test_pred  = best_dt.predict(X_test)

best_dt_train_acc = balanced_accuracy_score(y_train, y_train_pred)
best_dt_valid_acc = balanced_accuracy_score(y_val, y_valid_pred)
best_dt_test_acc  = balanced_accuracy_score(y_test, y_test_pred)

print(f"Train Balanced Accuracy: {best_dt_train_acc:.4f}")
print(f"Validation Balanced Accuracy: {best_dt_valid_acc:.4f}")
print(f"Test Balanced Accuracy: {best_dt_test_acc:.4f}")


Train Balanced Accuracy: 0.8772
Validation Balanced Accuracy: 0.7324
Test Balanced Accuracy: 0.7489


In [23]:
# Use the pretty-print helper provided
pretty_print_sklearn_tree(best_dt, vocab)

The binary tree structure has 817 nodes.
- depth   0 has    1 nodes, of which    0 are leaves
- depth   1 has    2 nodes, of which    0 are leaves
- depth   2 has    4 nodes, of which    0 are leaves
- depth   3 has    8 nodes, of which    0 are leaves
- depth   4 has   16 nodes, of which    6 are leaves
- depth   5 has   20 nodes, of which    5 are leaves
- depth   6 has   30 nodes, of which   14 are leaves
- depth   7 has   32 nodes, of which   18 are leaves
- depth   8 has   28 nodes, of which   12 are leaves
- depth   9 has   32 nodes, of which   11 are leaves
- depth  10 has   42 nodes, of which   21 are leaves
- depth  11 has   42 nodes, of which   23 are leaves
- depth  12 has   38 nodes, of which   17 are leaves
- depth  13 has   42 nodes, of which   21 are leaves
- depth  14 has   42 nodes, of which   25 are leaves
- depth  15 has   34 nodes, of which   18 are leaves
- depth  16 has   32 nodes, of which   14 are leaves
- depth  17 has   36 nodes, of which   23 are leaves
- dep

### Interpret the best decision tree

The best decision tree (max_depth = 32, min_samples_leaf = 3) learns a relatively deep hierarchy of word-based rules.

Internal nodes typically split on high-sentiment words such as “great”, “bad”, “ecxel”, and “poor”. But it becomes increasingly granular deeper in the tree, using rare word pairs that lead to leaf nodes containing only a single training example. 

Many of these nodes predict similar sentiments, suggesting that the splits do not meaningfully improve classification. Overall, the tree appears to memorize the training data rather than learn robust patterns, making it prone to high variance.
This situation makes sense when the split variable doesn’t significantly change the class distribution  it still reduces impurity slightly (Gini gain), so it’s retained, but both child leaves predict the same class. This is a case of overfitting.

# Problem 2: Random forest

## 2A: Train a random forest with default settings

In [24]:
simple_forest = sklearn.ensemble.RandomForestClassifier(
    n_estimators=100,
    criterion='gini',
    max_features='sqrt',
    max_depth=3,
    min_samples_leaf=1,
    min_samples_split=2,
    random_state=101
    )

### Fit the forest

**TODO Train on the training set** in the next coding cell

In [25]:
simple_forest.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,3
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [26]:
simple_rf_train_acc  = balanced_accuracy_score(y_train, simple_forest.predict(X_train))
simple_rf_valid_acc  = balanced_accuracy_score(y_val, simple_forest.predict(X_val))
simple_rf_test_acc   = balanced_accuracy_score(y_test, simple_forest.predict(X_test))

print(f"Simple RF Train Balanced Accuracy: {simple_rf_train_acc:.4f}")
print(f"Simple RF Valid Balanced Accuracy: {simple_rf_valid_acc:.4f}")
print(f"Simple RF Test Balanced Accuracy: {simple_rf_test_acc:.4f}")

Simple RF Train Balanced Accuracy: 0.8191
Simple RF Valid Balanced Accuracy: 0.7972
Simple RF Test Balanced Accuracy: 0.7782


## 2B & Table 2: Feature Importances

In [27]:
importances = simple_forest.feature_importances_

In [28]:
# Sort features by importance
indices = np.argsort(importances)[::-1]  # descending order

top10_indices = indices[:10]
top10_words = [vocab[i] for i in top10_indices]
top10_importances = importances[top10_indices]


In [29]:
low_importance_indices = np.where(importances < 0.00001)[0]
num_low_importance = len(low_importance_indices)

# Randomly pick 10 if available
np.random.seed(101)
random_low10_indices = np.random.choice(low_importance_indices, size=min(10, num_low_importance), replace=False)

random_low10_words = [vocab[i] for i in random_low10_indices]
random_low10_importances = importances[random_low10_indices]


<!-- ### Table 2
**Sample Output** (Feel free to print all words and organize them in any software)

|**Important Words**|**Unimportant Words**|
|:-:|:-:|
|I1 |  U1  |
|I2 |  U2  |
|I3 |  U3  |
|I4 |  U4  |
|I5 |  U5  |
|I6 |  U6  |
|I7 |  U7  |
|I8 |  U8  |
|I9 |  U9  |
|I0 |  U0  | -->

In [30]:
top10_df = pd.DataFrame({
    'Top 10 Important Words': top10_words,
    'Importance': top10_importances
})

low10_df = pd.DataFrame({
    'Low-Importance Words': random_low10_words,
    'Importance': random_low10_importances
})

display(pd.concat([top10_df, low10_df], axis=1))
print(f"\nNumber of terms with near-zero importance (< 0.00001): {num_low_importance}")

,Top 10 Important Words,Importance,Low-Importance Words,Importance
0,return,0.032990,doing_the,0.0
1,excel,0.029485,bought_these,0.0
2,great,0.028984,mix,0.0
3,worst,0.028407,not_all,0.0
4,poor,0.026748,vision,0.0
5,disappoint,0.024952,example_the,0.0
6,your_money,0.018002,the_author's,0.0
7,i_love,0.017955,abuse,0.0
8,the_best,0.017662,went_on,0.0
9,don't,0.017654,also_very,0.0



Number of terms with near-zero importance (< 0.00001): 7296


In [31]:
y_pred = simple_forest.predict(X_val)
acc = balanced_accuracy_score(y_val, y_pred)
print(f"Balanced Accuracy (Random Forest): {acc:.4f}")


Balanced Accuracy (Random Forest): 0.7972


## 2C: Best Random Forest via grid search

In [32]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

# Define the base classifier
rf = RandomForestClassifier(
    n_estimators=100,
    criterion='gini',
    min_samples_leaf=1,
    random_state=101
)

# Define grid
param_grid = {
    'max_features': [3, 10, 33, 100, 333],
    'max_depth': [16, 32],
}

# Run grid search
grid_search_rf = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    scoring='balanced_accuracy',
    cv=3,
    n_jobs=-1,
    verbose=1
)

### Do the search!

In [33]:
grid_search_rf.fit(X_train, y_train)

Fitting 3 folds for each of 10 candidates, totalling 30 fits


,estimator,RandomForestC...dom_state=101)
,param_grid,"{'max_depth': [16, 32], 'max_features': [3, 10, ...]}"
,scoring,'balanced_accuracy'
,n_jobs,-1
,refit,True
,cv,3
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,100


### Display search results

In [34]:
print("Best Parameters:", grid_search_rf.best_params_)
print("Best Balanced Accuracy:", grid_search_rf.best_score_)

Best Parameters: {'max_depth': 32, 'max_features': 33}
Best Balanced Accuracy: 0.8320335405216213


### Build the best random forest using the best hyperparameters found in 2B 

This is necessary so you have the specific best performing forest in your workspace.

Train *only* on training set (do not merge train and valid)


In [35]:
best_params = grid_search_rf.best_params_
print(best_params)

{'max_depth': 32, 'max_features': 33}


In [36]:
best_forest = RandomForestClassifier(
    n_estimators=100,            # fixed as per problem
    criterion='gini',
    max_features=best_params['max_features'],
    max_depth=best_params['max_depth'],
    min_samples_leaf=1,
    random_state=101
)

# Train only on training data
best_forest.fit(X_train, y_train)


,n_estimators,100
,criterion,'gini'
,max_depth,32
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,33
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [37]:
y_train_pred = best_forest.predict(X_train)
y_val_pred = best_forest.predict(X_val)
y_test_pred  = best_forest.predict(X_test)

best_rf_train_acc    = balanced_accuracy_score(y_train, best_forest.predict(X_train))
best_rf_valid_acc    = balanced_accuracy_score(y_val, best_forest.predict(X_val))
best_rf_test_acc     = balanced_accuracy_score(y_test, best_forest.predict(X_test))

print("Train Balanced Accuracy:", best_rf_train_acc)
print("Validation Balanced Accuracy:", best_rf_valid_acc)
print("Test Balanced Accuracy:", best_rf_test_acc)

Train Balanced Accuracy: 0.9644019000936142
Validation Balanced Accuracy: 0.8512044503419414
Test Balanced Accuracy: 0.8369334097275274


In [38]:
# # Feature importances with vocabulary
# importances = best_forest.feature_importances_
# top_features = np.argsort(importances)[::-1][:10]

# print("Top 10 important words:")
# for i in top_features:
#     print(vocab[i], ":", importances[i])


# Random Forest Hyperparameter Analysis


In [39]:

# 1. What is the value of max_features of your best forest?
print("1. What is the value of max_features of your best forest?")
print(f"The best Random Forest model had: {best_params['max_features']} as the value for max_features.")

# Additional analysis you might want to include:
print(f"\n2. Complete best hyperparameters:")
for param, value in best_params.items():
    print(f"   {param}: {value}")

print(f"\n3. Best cross-validation score: {grid_search_rf.best_score_:.4f}")

# Interpretation of max_features value
max_feat = best_params['max_features']
total_features = X_train.shape[1]
print(f"\n4. Feature selection analysis:")
print(f"   Total features available: {total_features}")
print(f"   Features considered per split: {max_feat}")
print(f"   Percentage of features used: {(max_feat/total_features)*100:.1f}%")

# Explain what this means
if max_feat == 3:
    print("   This is a very restrictive feature selection - only 3 out of 7729 features per split.")
elif max_feat <= 100:
    print("   This represents a moderate feature selection strategy.")
else:
    print("   This uses a relatively large number of features per split.")

1. What is the value of max_features of your best forest?
The best Random Forest model had: 33 as the value for max_features.

2. Complete best hyperparameters:
   max_depth: 32
   max_features: 33

3. Best cross-validation score: 0.8320

4. Feature selection analysis:
   Total features available: 7729
   Features considered per split: 33
   Percentage of features used: 0.4%
   This represents a moderate feature selection strategy.


### 5. What is the maximum possible value for max_features for this dataset? Why is it beneficial to tune this hyperparameter?

The maximum possible value for max_features is equal to the total number of features (i.e., the size of the vocabulary) used in the dataset.

In text-based datasets like this one, the vocabulary size can be very large (thousands of features).

Tuning max_features helps control how many features are randomly considered at each split:
- **Lower values** increase randomness and model diversity (reducing overfitting)
- **Higher values** allow trees to consider more features (increasing accuracy but potentially reducing diversity)

Thus, adjusting max_features balances the tradeoff between **bias** and **variance**, improving overall generalization.

### 6. What is the primary tradeoff controlled by the n_estimators hyperparameter? Can you overfit by setting it too large? Why or why not?

The n_estimators parameter controls the **number of trees** in the forest.

- Increasing n_estimators generally reduces **variance** and improves stability of predictions, as more trees average out random fluctuations
- However, it also **increases computational cost** and training time

We **cannot overfit** merely by setting n_estimators too large, because Random Forests rely on **bagging (bootstrap aggregation)** i.e. each tree is trained on a random subset of data and features.

After a certain point, adding more trees provides **diminishing returns** in accuracy while only increasing computation.

So:
- More trees → smoother, more stable predictions
- But too many trees → slower training and inference, not necessarily higher accuracy

### Table 3: Comparison of methods on the bag-of-words to sentiment classification task.

<!-- `|**method**|**max depth**|**num trees**|**train BAcc**|**valid BAcc**|**test BAcc**|
|:-|:-:|:-:|:-:|:-:|:-:|
|simple Tree	| 1 | 1 | 0.123	|0.456	|0.890|
|best Tree	|1 | 1 | 0.123	|0.456	|0.890|
|simple RandomForest	|1 | 1 | 0.123	|0.456	|0.890|
|best RandomForest	|1 | 1 | 0.123	|0.456	|0.890|` -->

In [40]:
results_table = pd.DataFrame([
    ["Simple DT", 3, 1, simple_dt_train_acc, simple_dt_valid_acc, simple_dt_test_acc],
    ["Best DT", best_dt.get_params()['max_depth'], 1, best_dt_train_acc, best_dt_valid_acc, best_dt_test_acc],
    ["Simple RF", 3, 100, simple_rf_train_acc, simple_rf_valid_acc, simple_rf_test_acc],
    ["Best RF", best_forest.get_params()['max_depth'], 100, best_rf_train_acc, best_rf_valid_acc, best_rf_test_acc]
], columns=["method", "max depth", "num trees", "train BAcc", "valid BAcc", "test BAcc"])

# Format the numeric columns to 3 decimal places
results_table["train BAcc"] = results_table["train BAcc"].round(3)
results_table["valid BAcc"] = results_table["valid BAcc"].round(3)
results_table["test BAcc"] = results_table["test BAcc"].round(3)

print(results_table.to_string(index=False))

   method  max depth  num trees  train BAcc  valid BAcc  test BAcc
Simple DT          3          1       0.646       0.645      0.646
  Best DT         32          1       0.877       0.732      0.749
Simple RF          3        100       0.819       0.797      0.778
  Best RF         32        100       0.964       0.851      0.837


### Overall Performance:

All models achieve balanced accuracies above 0.6, indicating that they learn meaningful patterns beyond random guessing. However, deeper models tend to overfit the training data, as reflected by the gap between training and validation/test accuracies.

### Decision Trees:

Increasing depth from 3 to 32 significantly boosts training accuracy (0.65 → 0.88), showing improved fitting capacity.
Validation and test accuracies also rise to around 0.74, indicating that deeper trees capture more structure in the data.
However, the larger train–test gap (~0.13) suggests moderate overfitting.

### Random Forests:

Even shallow forests outperform single trees, showing that bagging reduces variance and improves stability.

The deep forest (depth=32) achieves the highest training and test accuracies (~0.964 / 0.851), demonstrating strong learning capacity and good generalization due to *ensemble averaging*.